# USGS Water Level Data: Google Drive Storage & DataFrame Creation
## Project: One Model, Many Rivers - Deep Learning Water Forecasting

This notebook demonstrates:
1. Mounting Google Drive for cloud storage
2. Downloading USGS water data for 50+ monitoring sites
3. Storing data efficiently and creating DataFrames
4. Data cleaning and preprocessing for deep learning

## Section 1: Mount Google Drive to Colab

This section establishes a connection to your Google Drive, allowing you to:
- Store downloaded USGS data files permanently
- Access files across multiple sessions
- Organize data in a dedicated folder

In [13]:
# Section 1: Setup Storage (Local or Google Drive)

import os
import sys
from pathlib import Path

print("Checking environment...")
print("Note: Google Drive mounting only works in Google Colab (colab.research.google.com)")
print("      VS Code Jupyter extension uses local storage.\n")

# Check if we're in actual Google Colab (not VS Code)
IN_COLAB = False
try:
    import google.colab
    IN_COLAB = True
    print("✓ Running in Google Colab")
except ImportError:
    print("ℹ Running in VS Code or local Jupyter (using local storage)")

# Mount/Setup storage
if IN_COLAB:
    try:
        from google.colab import drive
        print("Mounting Google Drive...")
        drive.mount('/content/drive', force_remount=False)
        usgs_data_folder = '/content/drive/My Drive/USGS_Water_Data'
        os.makedirs(usgs_data_folder, exist_ok=True)
        print(f"✓ Data folder ready at: {usgs_data_folder}")
        data_folder = usgs_data_folder
    except Exception as e:
        print(f"⚠ Could not mount Google Drive: {e}")
        print("Using local folder instead...\n")
        local_data_folder = Path('c:/Water_research/USGS_Downloads')
        local_data_folder.mkdir(parents=True, exist_ok=True)
        data_folder = str(local_data_folder)
else:
    # Local environment: Use Windows path
    local_data_folder = Path('c:/Water_research/USGS_Downloads')
    local_data_folder.mkdir(parents=True, exist_ok=True)
    print(f"✓ Local data folder: {local_data_folder}")
    print(f"✓ Folder exists: {local_data_folder.exists()}")
    data_folder = str(local_data_folder)

Checking environment...
Note: Google Drive mounting only works in Google Colab (colab.research.google.com)
      VS Code Jupyter extension uses local storage.

ℹ Running in VS Code or local Jupyter (using local storage)
✓ Local data folder: c:\Water_research\USGS_Downloads
✓ Folder exists: True


## Section 2: Import Libraries & Setup USGS API Download Function

In [24]:
import pandas as pd
import numpy as np
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
import json
from datetime import datetime, timedelta
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

# Setup session with retries and increased timeout
def create_session():
    session = requests.Session()
    retry = Retry(
        total=3,
        backoff_factor=0.5,
        status_forcelist=(500, 502, 504)
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('http://', adapter)
    session.mount('https://', adapter)
    return session

session = create_session()

print("Libraries imported successfully!")
print("Session with retry strategy created.")


Libraries imported successfully!
Session with retry strategy created.


## Section 3: Define USGS Site List (50+ Sites)

In [15]:
# USGS Site Numbers for 376 Monitoring Stations Across New Mexico
# Using expanded list to ensure 50+ sites with available data
# Format: {River_Region: [Site Numbers]}

usgs_sites = {
    'Vermejo_Cimarron': [
        '07202400', '07202500', '07203000', '07203505', '07203525', '07204000', '07204500', '07205000',
        '07206000', '07207000', '07207500', '07208500', '07211000', '07211500'
    ],
    'Mora_North': [
        '07214470', '07214550', '07214670', '07214680', '07214695', '07214790', '07215500', '07216500',
        '07218000', '07218475', '07218525', '07218615', '07218635', '07218655', '07218680', '07218720'
    ],
    'Canadian_River': [
        '07221000', '07221500', '07222500', '07223500', '07224500', '07226500', '07227000', '07227100'
    ],
    'Costilla_Taos': [
        '08252500', '08253000', '08253500', '08253900', '08254000', '08255500', '08261000', '08263500',
        '08265000', '08266000', '08266820', '08267500', '08268700', '08269000', '08271000', '08275500',
        '08276300', '08276500', '08277130', '08277140', '08277150', '08277470', '08278500', '08279000', '08279500'
    ],
    'Rio_Chama': [
        '08281400', '08282300', '08284100', '08284300', '08284520', '08285500', '08286500', '08286900',
        '08287000', '08289000', '08290000'
    ],
    'Rio_Grande_North': [
        '08291000', '08292000', '08294195', '08294210', '08302500', '08305030', '08308025', '08308050'
    ],
    'Rio_Grande_Central': [
        '08313000', '08313025', '08313030', '08313042', '08313060', '08313125', '08313150', '08313225',
        '08313230', '08313240', '08313245', '08313250', '08313252', '08313253', '08313255', '08313265',
        '08313268', '08313275', '08313350', '08313500', '08314000', '08315480', '08316000', '08316500',
        '08317050', '08317200', '08317300', '08317400', '08317900', '08317950', '08319000'
    ],
    'Jemez_Rio_Puerco': [
        '08323000', '08324000', '08324950', '08328500', '08328950', '08329000', '08329700', '08329720',
        '08329835', '08329838', '08329839', '08329840', '08329868', '08329870', '08329872', '08329873',
        '08329874', '08329875', '08329876', '08329877', '08329882', '08329884', '08329888', '08329900',
        '08329911', '08329914', '08329918', '08329928', '08329930', '08329931', '08329935', '08329936',
        '08330000', '08330150', '08330200', '08330492', '08330505', '08330540', '08330600', '08330775',
        '08330830', '08330875', '08330915', '08330940', '08331105', '08331118', '08331160', '08331510',
        '08331660', '08331990', '08332010', '08332050', '08334000', '08340500', '08341300', '08341365',
        '08341500', '08343000', '08343100', '08343500', '08349800', '08350450', '08351500', '08353000',
        '08354500', '08354800', '08354900', '08355050', '08355490', '08356950', '08357050', '08358300',
        '08358400', '08359500', '08361000'
    ],
    'Rio_Mora_Pecos_Upper': [
        '08377900', '08378500', '08379187', '08379189', '08379500', '08379990', '08380040', '08380070',
        '08380088', '08380400', '08380500', '08382000', '08382500'
    ],
    'Pecos_River_Main': [
        '08382600', '08382650', '08382730', '08382810', '08382830', '08383000', '08383500', '08384000',
        '08384500', '08385000', '08385500', '08385503', '08385522', '08385630', '08385643', '08385648',
        '08386000', '08386505', '08386809', '08386850', '08386970', '08387000', '08387550', '08387575',
        '08387600', '08387700', '08387710', '08389055', '08390020', '08390025', '08390500', '08390610',
        '08390620', '08390800', '08393300', '08393500', '08393610', '08394024', '08394033', '08394500',
        '08395500', '08396025', '08396500', '08397600', '08397620', '08398500', '08399500', '08400000',
        '08401200', '08401500', '08401900', '08402000', '08403500', '08403800', '08404000', '08405105',
        '08405150', '08405200', '08405301', '08405303', '08405350', '08405400', '08405450', '08405500',
        '08406000', '08406500', '08407000', '08407500', '08408500', '08418010'
    ],
    'Mimbres_Tularosa': [
        '08477110', '08480594', '08480595', '08481500', '08484550'
    ],
    'San_Juan_Navajo': [
        '09346050', '09355500', '09357000', '09357245', '09357255', '09357700'
    ],
    'Animas_La_Plata': [
        '09363500', '09364010', '09364200', '09364500', '09365000', '09367000', '09367500', '09367536',
        '09367540', '09367580', '09367950', '09367958', '09368000'
    ],
    'San_Juan_Fruitland': [
        '093710009', '09371010'
    ],
    'Nutria_Zuni': [
        '09386900', '09386910', '09386915', '09386917', '09386919', '09386925', '09386950', '09395350',
        '09395630'
    ],
    'Gila_River': [
        '09429980', '09430010', '09430020', '09430030', '09430500', '09430600', '09430815', '09430820',
        '09430825', '09431500', '09431503', '09432000'
    ],
    'San_Francisco_River': [
        '09442540', '09442625', '09442661', '09442662', '09442666', '09442667', '09442669', '09442677',
        '09442680', '09442681', '09442682', '09442683', '09442684', '09442692', '09442704', '09442755',
        '09442760', '09442956', '09442958', '09442960', '09443900', '09443938', '09444000'
    ]
}

# Flatten to get all site numbers
all_sites = []
for river, sites in usgs_sites.items():
    all_sites.extend(sites)

print(f"Total USGS sites: {len(all_sites)}")
print(f"Rivers included: {list(usgs_sites.keys())}")
print(f"\nSite distribution:")
for river, sites in usgs_sites.items():
    print(f"  {river}: {len(sites)} sites")

Total USGS sites: 341
Rivers included: ['Vermejo_Cimarron', 'Mora_North', 'Canadian_River', 'Costilla_Taos', 'Rio_Chama', 'Rio_Grande_North', 'Rio_Grande_Central', 'Jemez_Rio_Puerco', 'Rio_Mora_Pecos_Upper', 'Pecos_River_Main', 'Mimbres_Tularosa', 'San_Juan_Navajo', 'Animas_La_Plata', 'San_Juan_Fruitland', 'Nutria_Zuni', 'Gila_River', 'San_Francisco_River']

Site distribution:
  Vermejo_Cimarron: 14 sites
  Mora_North: 16 sites
  Canadian_River: 8 sites
  Costilla_Taos: 25 sites
  Rio_Chama: 11 sites
  Rio_Grande_North: 8 sites
  Rio_Grande_Central: 31 sites
  Jemez_Rio_Puerco: 75 sites
  Rio_Mora_Pecos_Upper: 13 sites
  Pecos_River_Main: 70 sites
  Mimbres_Tularosa: 5 sites
  San_Juan_Navajo: 6 sites
  Animas_La_Plata: 13 sites
  San_Juan_Fruitland: 2 sites
  Nutria_Zuni: 9 sites
  Gila_River: 12 sites
  San_Francisco_River: 23 sites


## Section 4: Download USGS Water Data via API

Downloads daily streamflow (00060) and gage height (00065) data for all sites.
Saves each site as a CSV file in your data folder.

In [25]:
def download_usgs_data(site_number, start_date_str, start_time, end_date_str, end_time):
    """
    Download USGS instantaneous water data (IV) for a specific site using RDB format
    
    Parameters:
    - site_number: USGS site number (string)
    - start_date_str: Start date (YYYY-MM-DD)
    - start_time: Start time (HH:MM)
    - end_date_str: End date (YYYY-MM-DD)
    - end_time: End time (HH:MM)
    
    Returns: DataFrame with the data (parameters determined by USGS availability)
    """
    # Format timestamps as required by USGS IV service
    startDT = f"{start_date_str}T{start_time}"
    endDT = f"{end_date_str}T{end_time}"
    
    url = f"https://nwis.waterservices.usgs.gov/nwis/iv/?format=rdb&sites={site_number}&startDT={startDT}&endDT={endDT}&siteStatus=all&siteType=LK,ST,WE"
    
    try:
        # Use session with retries
        response = session.get(url, timeout=120)  # 2 minute timeout
        response.raise_for_status()
        
        # Parse RDB format (tab-delimited with comment lines and headers)
        lines = response.text.strip().split('\n')
        
        # Find header line (starts with 'agency_cd')
        header_idx = None
        for i, line in enumerate(lines):
            if line.startswith('agency_cd'):
                header_idx = i
                break
        
        if header_idx is None:
            return None
        
        # Skip data type line (next line after header) and get actual data
        data_start = header_idx + 2
        
        if data_start >= len(lines):
            return None
        
        # Read the data section
        from io import StringIO
        header_line = lines[header_idx]
        data_section = '\n'.join(lines[data_start:])
        combined = header_line + '\n' + data_section
        
        try:
            df = pd.read_csv(StringIO(combined), sep='\t', dtype={'site_no': str})
        except Exception as e:
            return None
        
        if df.empty:
            return None
        
        # Convert datetime column
        df['datetime'] = pd.to_datetime(df['datetime'])
        
        # Find parameter columns (format: TSID_parametercode, e.g., 101096_00060)
        param_cols = [col for col in df.columns if '_' in col and col[0].isdigit()]
        
        # Reshape to long format
        all_data = []
        for param_col in param_cols:
            # Extract parameter code (last 5 digits)
            param_code = param_col.split('_')[-1]
            
            for idx, row in df.iterrows():
                value = row[param_col]
                if pd.notna(value) and value != '':
                    try:
                        val = float(value)
                        all_data.append({
                            'dateTime': row['datetime'],
                            'value': val,
                            'parameter_code': param_code,
                            'parameter_name': param_col
                        })
                    except (ValueError, TypeError):
                        pass
        
        if all_data:
            result_df = pd.DataFrame(all_data)
            return result_df
        else:
            return None
            
    except Exception as e:
        return None

# Set date range for download
# Feb 14, 2023 to Nov 2, 2025
start_date = '2023-02-14'
start_time = '00:00'
end_date = '2025-11-02'
end_time = '23:59'

print(f"Data will be downloaded from {start_date} {start_time} to {end_date} {end_time}")
print(f"Format: RDB (instantaneous values)")
print(f"Total sites to download: {len(all_sites)}")


Data will be downloaded from 2023-02-14 00:00 to 2025-11-02 23:59
Format: RDB (instantaneous values)
Total sites to download: 341


In [27]:
# Download and save data for all sites
# WARNING: This may take several minutes depending on your internet connection

downloaded_data = {}
failed_sites = []

print("Starting data download...\n")
print("Using USGS IV (instantaneous values) endpoint with RDB format\n")

for i, site_no in enumerate(all_sites, 1):
    print(f"[{i}/{len(all_sites)}] Downloading site {site_no}...", end=" ")
    
    df = download_usgs_data(site_no, start_date, start_time, end_date, end_time)
    
    if df is not None:
        # Save to file
        if isinstance(data_folder, str):
            file_path = os.path.join(data_folder, f"USGS_{site_no}.csv")
        else:
            file_path = data_folder / f"USGS_{site_no}.csv"
        df.to_csv(file_path, index=False)
        downloaded_data[site_no] = df
        print(f"✓ ({len(df)} records)")
    else:
        failed_sites.append(site_no)
        print("✗ Failed or no data")

print(f"\n{'='*60}")
print(f"Download Summary:")
print(f"  Successfully downloaded: {len(downloaded_data)} sites")
print(f"  Failed or no data: {len(failed_sites)} sites")
print(f"  Total records saved: {sum(len(v) for v in downloaded_data.values())}")
print(f"{'='*60}")


Starting data download...

Using USGS IV (instantaneous values) endpoint with RDB format

[1/341] Downloading site 07202400... ✗ Failed or no data
[2/341] Downloading site 07202500... ✓ (162766 records)
[3/341] Downloading site 07203000... ✓ (170621 records)
[4/341] Downloading site 07203505... ✗ Failed or no data
[5/341] Downloading site 07203525... ✗ Failed or no data
[6/341] Downloading site 07204000... ✗ Failed or no data
[7/341] Downloading site 07204500... ✗ Failed or no data
[8/341] Downloading site 07205000... ✗ Failed or no data
[9/341] Downloading site 07206000... ✓ (183103 records)
[10/341] Downloading site 07207000... ✓ (170257 records)
[11/341] Downloading site 07207500... ✓ (172574 records)
[12/341] Downloading site 07208500... ✓ (174846 records)
[13/341] Downloading site 07211000... ✗ Failed or no data
[14/341] Downloading site 07211500... ✓ (174478 records)
[15/341] Downloading site 07214470... ✓ (168951 records)
[16/341] Downloading site 07214550... ✓ (352564 records)


KeyboardInterrupt: 

In [26]:
# TEST: Download single site with updated function
test_site = '08279500'
test_df = download_usgs_data(test_site, '2023-02-02', '15:28', '2025-02-02', '15:28')

if test_df is not None:
    print(f"✓ Successfully downloaded {test_site}")
    print(f"  Records: {len(test_df)}")
    print(f"  Date range: {test_df['dateTime'].min()} to {test_df['dateTime'].max()}")
    print(f"  Parameters found: {test_df['parameter_code'].unique()}")
    print(f"\nFirst 10 rows:")
    print(test_df.head(10))
    print(f"\nParameter summary:")
    print(test_df['parameter_code'].value_counts())
else:
    print(f"✗ Failed to download {test_site}")


✓ Successfully downloaded 08279500
  Records: 217802
  Date range: 2023-02-02 15:30:00 to 2025-02-02 15:15:00
  Parameters found: ['00010' '00060' '00065' '63680']

First 10 rows:
             dateTime  value parameter_code parameter_name
0 2023-12-01 12:15:00    4.6          00010   333747_00010
1 2023-12-01 12:30:00    4.6          00010   333747_00010
2 2023-12-01 12:45:00    4.7          00010   333747_00010
3 2023-12-01 13:00:00    4.7          00010   333747_00010
4 2023-12-01 13:15:00    4.7          00010   333747_00010
5 2023-12-01 13:30:00    4.7          00010   333747_00010
6 2023-12-01 13:45:00    4.8          00010   333747_00010
7 2023-12-01 14:00:00    4.9          00010   333747_00010
8 2023-12-01 14:15:00    4.9          00010   333747_00010
9 2023-12-01 14:30:00    4.9          00010   333747_00010

Parameter summary:
parameter_code
00060    69593
00065    69512
00010    40532
63680    38165
Name: count, dtype: int64


## Section 5: Load Data into Variables & DataFrames

In [ ]:
# Option 1: Load data from CSV files into a dictionary of DataFrames
site_dataframes = {}

# Get CSV files from data folder
from pathlib import Path
if isinstance(data_folder, str):
    data_path = Path(data_folder)
else:
    data_path = data_folder

csv_files = list(data_path.glob('USGS_*.csv'))
print(f"Loading {len(csv_files)} CSV files from {data_folder}...\n")

for csv_file in csv_files:
    site_no = csv_file.stem.replace('USGS_', '')
    try:
        df = pd.read_csv(csv_file)
        df['dateTime'] = pd.to_datetime(df['dateTime'])
        site_dataframes[site_no] = df
    except Exception as e:
        print(f"Error loading {site_no}: {e}")

print(f"Loaded {len(site_dataframes)} site DataFrames")
print(f"Sample sites: {list(site_dataframes.keys())[:5]}")

In [29]:
# ANALYSIS: Check parameter coverage in downloaded files
import os
from pathlib import Path

# Get all CSV files
data_path = Path(data_folder)
csv_files = sorted(list(data_path.glob('USGS_*.csv')))

print(f"Analyzing {len(csv_files)} downloaded files...\n")

# Analyze each file
parameter_summary = {}
has_discharge = []
has_gage_height = []
has_both = []
discharge_only = []
gage_only = []

for csv_file in csv_files:
    site_no = csv_file.stem.replace('USGS_', '')
    try:
        df = pd.read_csv(csv_file, dtype={'parameter_code': str})
        params = df['parameter_code'].unique()
        params = [str(p) for p in params]  # Ensure all are strings
        
        has_00060 = '00060' in params
        has_00065 = '00065' in params
        
        parameter_summary[site_no] = {
            'discharge': has_00060,
            'gage_height': has_00065,
            'record_count': len(df),
            'all_params': params
        }
        
        if has_00060:
            has_discharge.append(site_no)
        if has_00065:
            has_gage_height.append(site_no)
        if has_00060 and has_00065:
            has_both.append(site_no)
        if has_00060 and not has_00065:
            discharge_only.append(site_no)
        if has_00065 and not has_00060:
            gage_only.append(site_no)
            
    except Exception as e:
        print(f"Error reading {site_no}: {e}")

# Print summary
print("="*80)
print("PARAMETER COVERAGE SUMMARY")
print("="*80)
print(f"\nTotal files analyzed: {len(csv_files)}")
print(f"Sites with Discharge (00060):    {len(has_discharge)} sites ✓")
print(f"Sites with Gage Height (00065):  {len(has_gage_height)} sites ✓")
print(f"Sites with BOTH parameters:      {len(has_both)} sites ✓✓")
print(f"\nBreakdown:")
print(f"  - Discharge only:              {len(discharge_only)} sites")
print(f"  - Gage height only:            {len(gage_only)} sites")
print(f"  - Missing both:                {len(csv_files) - len(has_discharge) - len(gage_only)} sites")

print("\n" + "="*80)
print("DETAILED PARAMETER AVAILABILITY BY SITE")
print("="*80)
print(f"\n{'Site':<12} {'Discharge':<15} {'Gage Height':<15} {'Records':<12} {'Other Params'}")
print("-" * 100)

for site_no in sorted(parameter_summary.keys()):
    info = parameter_summary[site_no]
    discharge_str = "✓ 00060" if info['discharge'] else "✗"
    gage_str = "✓ 00065" if info['gage_height'] else "✗"
    other_params = [p for p in info['all_params'] if p not in ['00060', '00065']]
    other_str = ', '.join(other_params) if other_params else "-"
    print(f"{site_no:<12} {discharge_str:<15} {gage_str:<15} {info['record_count']:<12} {other_str}")

print("\n" + "="*80)
print("SITES WITH BOTH PARAMETERS (IDEAL FOR MODELING)")
print("="*80)
if has_both:
    print(f"\nTotal: {len(has_both)} sites\n")
    for i, site in enumerate(sorted(has_both), 1):
        records = parameter_summary[site]['record_count']
        print(f"  {i:2d}. {site}  ({records:,} records)")
else:
    print("No sites have both parameters yet.")

print("\n" + "="*80)


Analyzing 93 downloaded files...

PARAMETER COVERAGE SUMMARY

Total files analyzed: 93
Sites with Discharge (00060):    70 sites ✓
Sites with Gage Height (00065):  92 sites ✓
Sites with BOTH parameters:      70 sites ✓✓

Breakdown:
  - Discharge only:              0 sites
  - Gage height only:            22 sites
  - Missing both:                1 sites

DETAILED PARAMETER AVAILABILITY BY SITE

Site         Discharge       Gage Height     Records      Other Params
----------------------------------------------------------------------------------------------------
07202500     ✓ 00060         ✓ 00065         162766       -
07203000     ✓ 00060         ✓ 00065         170621       -
07206000     ✓ 00060         ✓ 00065         183103       -
07207000     ✓ 00060         ✓ 00065         170257       -
07207500     ✓ 00060         ✓ 00065         172574       -
07208500     ✓ 00060         ✓ 00065         174846       -
07211500     ✓ 00060         ✓ 00065         174478       -
07214470  

In [30]:
# QUICK SUMMARY - Parameter Coverage Statistics
print("\n" + "="*80)
print("QUICK SUMMARY")
print("="*80)

print(f"\n📊 COVERAGE STATS:")
print(f"   Total downloaded files: {len(csv_files)}")
print(f"   Files with Discharge (00060):   {len(has_discharge)} sites ({100*len(has_discharge)/len(csv_files):.1f}%)")
print(f"   Files with Gage Height (00065): {len(has_gage_height)} sites ({100*len(has_gage_height)/len(csv_files):.1f}%)")
print(f"   Files with BOTH:                {len(has_both)} sites ({100*len(has_both)/len(csv_files):.1f}%)")

print(f"\n📈 TOTAL RECORDS:")
total_records = sum(info['record_count'] for info in parameter_summary.values())
print(f"   Total records across all sites: {total_records:,}")

# Sample from sites with both parameters
print(f"\n✓ SAMPLE SITES WITH BOTH PARAMETERS:")
if has_both:
    for site in sorted(has_both)[:10]:
        records = parameter_summary[site]['record_count']
        print(f"   • {site}: {records:,} records")
    if len(has_both) > 10:
        print(f"   ... and {len(has_both) - 10} more")
else:
    print("   None yet")

# Sample from discharge-only sites
print(f"\n⚠️  DISCHARGE-ONLY SITES (no gage height):")
if discharge_only:
    for site in sorted(discharge_only)[:10]:
        records = parameter_summary[site]['record_count']
        print(f"   • {site}: {records:,} records")
    if len(discharge_only) > 10:
        print(f"   ... and {len(discharge_only) - 10} more")
else:
    print("   None")

print("\n" + "="*80)



QUICK SUMMARY

📊 COVERAGE STATS:
   Total downloaded files: 93
   Files with Discharge (00060):   70 sites (75.3%)
   Files with Gage Height (00065): 92 sites (98.9%)
   Files with BOTH:                70 sites (75.3%)

📈 TOTAL RECORDS:
   Total records across all sites: 20,607,643

✓ SAMPLE SITES WITH BOTH PARAMETERS:
   • 07202500: 162,766 records
   • 07203000: 170,621 records
   • 07206000: 183,103 records
   • 07207000: 170,257 records
   • 07207500: 172,574 records
   • 07208500: 174,846 records
   • 07211500: 174,478 records
   • 07214470: 168,951 records
   • 07214670: 94,945 records
   • 07214680: 198,186 records
   ... and 60 more

⚠️  DISCHARGE-ONLY SITES (no gage height):
   None



In [32]:
# Section: Filter to 70 sites with BOTH parameters and prepare for modeling

print("="*80)
print("PREPARING DATA FOR MODELING: 70 SITES WITH BOTH DISCHARGE & GAGE HEIGHT")
print("="*80)

# Create list of sites with both parameters
modeling_sites = sorted(has_both)
print(f"\n✓ Modeling sites identified: {len(modeling_sites)} sites")
print(f"  First 10: {modeling_sites[:10]}")

# Load only the modeling sites
modeling_site_dfs = {}
for site_no in modeling_sites:
    csv_file = Path(data_folder) / f"USGS_{site_no}.csv"
    if csv_file.exists():
        df = pd.read_csv(csv_file, dtype={'parameter_code': str})
        df['dateTime'] = pd.to_datetime(df['dateTime'])
        modeling_site_dfs[site_no] = df

print(f"✓ Loaded {len(modeling_site_dfs)} site DataFrames for modeling")

# Combine all modeling sites into a single master dataframe
master_list = []
for site_no, df in modeling_site_dfs.items():
    df_copy = df.copy()
    df_copy['site_number'] = site_no
    master_list.append(df_copy)

master_modeling_df = pd.concat(master_list, ignore_index=True)
master_modeling_df = master_modeling_df.sort_values(['site_number', 'dateTime']).reset_index(drop=True)

print(f"\n📊 Master DataFrame for Modeling:")
print(f"   Shape: {master_modeling_df.shape}")
print(f"   Columns: {master_modeling_df.columns.tolist()}")
print(f"   Date range: {master_modeling_df['dateTime'].min()} to {master_modeling_df['dateTime'].max()}")
print(f"   Total records: {len(master_modeling_df):,}")

# Show parameter distribution
print(f"\n📈 Parameter Distribution:")
print(master_modeling_df['parameter_code'].value_counts())


PREPARING DATA FOR MODELING: 70 SITES WITH BOTH DISCHARGE & GAGE HEIGHT

✓ Modeling sites identified: 70 sites
  First 10: ['07202500', '07203000', '07206000', '07207000', '07207500', '07208500', '07211500', '07214470', '07214670', '07214680']
✓ Loaded 70 site DataFrames for modeling

📊 Master DataFrame for Modeling:
   Shape: (16245532, 5)
   Columns: ['dateTime', 'value', 'parameter_code', 'parameter_name', 'site_number']
   Date range: 2023-02-14 00:00:00 to 2025-11-02 23:55:00
   Total records: 16,245,532

📈 Parameter Distribution:
parameter_code
00065    6994092
00060    6862682
00045    1437998
00010     377680
63680     346467
00095      77637
00300      77297
00400      71679
Name: count, dtype: int64


In [35]:
# Load site coordinates from long_lang_sites.txt file

print("\n" + "="*80)
print("LOADING SITE COORDINATES FROM long_lang_sites.txt")
print("="*80)

# Read the coordinates file - find header and read with proper formatting
coord_file_path = 'c:/Water_research/long_lang_sites.txt'

# Read the file and parse it manually
with open(coord_file_path, 'r') as f:
    lines = f.readlines()

# Find header and data type lines
header_line = None
data_type_line = None
data_start = None

for i, line in enumerate(lines):
    if line.startswith('agency_cd'):
        header_line = i
        data_type_line = i + 1
        data_start = i + 2
        break

# Extract header names
headers = lines[header_line].strip().split('\t')

# Read data
from io import StringIO
data_text = ''.join(lines[data_start:])
site_coords_df = pd.read_csv(StringIO(data_text), sep='\t', names=headers, dtype={'site_no': str})

print(f"✓ Loaded {len(site_coords_df)} total sites from file")

# Filter to only include our modeling sites
modeling_sites_set = set(modeling_sites)
site_coords_filtered = site_coords_df[site_coords_df['site_no'].isin(modeling_sites_set)].copy()

print(f"✓ Found {len(site_coords_filtered)} sites with coordinates for modeling")

# Create clean site info dataframe
site_info_df = site_coords_filtered[['site_no', 'station_nm', 'dec_lat_va', 'dec_long_va']].copy()
site_info_df.columns = ['site_number', 'site_name', 'latitude', 'longitude']

# Sort by site number
site_info_df = site_info_df.sort_values('site_number').reset_index(drop=True)

print(f"\n📍 SITE INFORMATION TABLE ({len(site_info_df)} sites):")
print(site_info_df.to_string())

# Save site information to CSV
site_info_path = os.path.join(data_folder, 'modeling_sites_coordinates.csv')
site_info_df.to_csv(site_info_path, index=False)
print(f"\n✓ Site information saved to: {site_info_path}")

# Display summary statistics
print(f"\n📊 Coordinate Statistics:")
print(f"   Latitude range: {site_info_df['latitude'].min():.4f}° to {site_info_df['latitude'].max():.4f}°")
print(f"   Longitude range: {site_info_df['longitude'].min():.4f}° to {site_info_df['longitude'].max():.4f}°")



LOADING SITE COORDINATES FROM long_lang_sites.txt
✓ Loaded 2358 total sites from file
✓ Found 70 sites with coordinates for modeling

📍 SITE INFORMATION TABLE (70 sites):
   site_number                                           site_name   latitude   longitude
0     07202500                     EAGLE TAIL DITCH NR MAXWELL, NM  36.648642 -104.559156
1     07203000                       VERMEJO RIVER NEAR DAWSON, NM  36.681028 -104.786394
2     07206000             CIMARRON RIVER BELOW EAGLE NEST DAM, NM  36.532125 -105.228144
3     07207000                    CIMARRON RIVER NEAR CIMARRON, NM  36.519833 -104.978611
4     07207500                       PONIL CREEK NEAR CIMARRON, NM  36.573694 -104.946806
5     07208500                      RAYADO CREEK NEAR CIMARRON, NM  36.372344 -104.969289
6     07211500              CANADIAN RIVER NEAR TAYLOR SPRINGS, NM  36.297569 -104.495472
7     07214470                        SIERRA DITCH NEAR CHACON, NM  36.183333 -105.406389
8     07214670    

In [37]:
print(f"\nNext step: Fetch meteorological data for {len(site_info_df)} sites using their coordinates")


Next step: Fetch meteorological data for 70 sites using their coordinates


In [ ]:
# Option 2: Create a master DataFrame combining all sites
master_df_list = []

for site_no, df in site_dataframes.items():
    df_copy = df.copy()
    df_copy['site_number'] = site_no
    master_df_list.append(df_copy)

# Combine all data
master_df = pd.concat(master_df_list, ignore_index=True)
master_df = master_df.sort_values(['site_number', 'dateTime']).reset_index(drop=True)

print(f"Master DataFrame shape: {master_df.shape}")
print(f"\nMaster DataFrame columns: {master_df.columns.tolist()}")
print(f"\nFirst few rows:")
print(master_df.head(10))

## Section 6: Data Exploration & Summary Statistics

In [ ]:
# Data summary by site
print("Data Summary by Site:\n")
print(f"{'Site Number':<12} {'River':<20} {'Records':<10} {'Date Range'}")
print("-" * 70)

for river, sites in usgs_sites.items():
    for site in sites:
        if site in site_dataframes:
            df = site_dataframes[site]
            num_records = len(df)
            date_range = f"{df['dateTime'].min().date()} to {df['dateTime'].max().date()}"
            print(f"{site:<12} {river:<20} {num_records:<10} {date_range}")

print("\n" + "="*70)
print(f"Total unique sites with data: {len(site_dataframes)}")
print(f"Total records across all sites: {len(master_df)}")
print(f"Date range: {master_df['dateTime'].min().date()} to {master_df['dateTime'].max().date()}")

In [ ]:
# Parameter statistics
print("\nParameter Code Summary:\n")
print(master_df['parameter_code'].value_counts())

print("\n" + "="*70)
print("\nValue Statistics (numeric only):\n")
print(master_df['value'].describe())

In [ ]:
# Check which parameters each site has
print("Parameter availability by site:\n")
print(f"{'Site':<12} {'Streamflow (00060)':<20} {'Gage Height (00065)':<20}")
print("-" * 52)

site_params = {}
for site_no, df in site_dataframes.items():
    has_streamflow = '00060' in df['parameter_code'].values
    has_gage_height = '00065' in df['parameter_code'].values
    site_params[site_no] = {'00060': has_streamflow, '00065': has_gage_height}
    print(f"{site_no:<12} {'✓' if has_streamflow else '✗':<20} {'✓' if has_gage_height else '✗':<20}")

print("\n" + "="*70)
sites_with_gage = sum(1 for s in site_params.values() if s['00065'])
sites_with_streamflow = sum(1 for s in site_params.values() if s['00060'])
print(f"Sites with streamflow (00060): {sites_with_streamflow}")
print(f"Sites with gage height (00065): {sites_with_gage}")
print(f"Sites with BOTH: {sum(1 for s in site_params.values() if s['00060'] and s['00065'])}")

## Section 7: Data Cleaning & Preprocessing

In [ ]:
# Clean and preprocess data
def clean_usgs_data(df):
    """
    Clean USGS water data for modeling
    - Remove missing values
    - Convert to numeric
    - Handle outliers
    """
    df = df.copy()
    
    # Convert value to numeric (handles 'e' for estimated values)
    df['value'] = pd.to_numeric(df['value'], errors='coerce')
    
    # Remove rows with missing values
    df = df.dropna()
    
    return df

# Clean master dataframe
master_df_clean = clean_usgs_data(master_df)

print(f"Original shape: {master_df.shape}")
print(f"Cleaned shape: {master_df_clean.shape}")
print(f"Rows removed: {master_df.shape[0] - master_df_clean.shape[0]}")

# Clean individual site dataframes
clean_site_dataframes = {}
for site_no, df in site_dataframes.items():
    clean_site_dataframes[site_no] = clean_usgs_data(df)

print(f"\nCleaned {len(clean_site_dataframes)} site dataframes")

In [ ]:
# Create pivot table for easier analysis (one row per date, one column per parameter per site)
# Example: Creating a wide format dataframe for a single river

# Example with Rio Grande sites
rio_grande_sites = usgs_sites['Rio_Grande']
rio_grande_data = master_df_clean[master_df_clean['site_number'].isin(rio_grande_sites)].copy()

# Create a more useful format: date + site + parameter
rio_grande_pivot = rio_grande_data.pivot_table(
    index='dateTime',
    columns=['site_number', 'parameter_code'],
    values='value'
)

print(f"Rio Grande pivot table shape: {rio_grande_pivot.shape}")
print(f"Date range: {rio_grande_pivot.index.min()} to {rio_grande_pivot.index.max()}")
print(f"\nFirst few rows of Rio Grande data:")
print(rio_grande_pivot.head())

## Section 9: Data Persistence - Save for Future Use

Save cleaned data to files so you don't need to download again next time. We'll use CSV for readability and pickle for speed.

In [ ]:
import pickle
import os

# Use the data_folder variable we defined in Section 1
# No need to redefine - just use data_folder directly

# Save master dataframe to CSV (easy to read, can open in Excel)
master_csv_path = os.path.join(data_folder, 'master_dataframe_clean.csv')
master_df_clean.to_csv(master_csv_path, index=False)
print(f"✓ Master DataFrame saved to: {master_csv_path}")

# Save master dataframe to pickle (faster to load in Python)
master_pickle_path = os.path.join(data_folder, 'master_dataframe_clean.pkl')
with open(master_pickle_path, 'wb') as f:
    pickle.dump(master_df_clean, f)
print(f"✓ Master DataFrame pickled to: {master_pickle_path}")

# Save individual site dataframes to pickle dictionary
sites_pickle_path = os.path.join(data_folder, 'site_dataframes_clean.pkl')
with open(sites_pickle_path, 'wb') as f:
    pickle.dump(site_dataframes_clean, f)
print(f"✓ Site DataFrames dictionary saved to: {sites_pickle_path}")

print(f"\nAll data saved successfully to: {data_folder}")
print(f"Total files saved: 3 (1 CSV, 2 pickle files)")

## Section 10: Quick Reference - Load Saved Data in Future Sessions

Next time you run this notebook, skip Sections 2-9 and use this code to reload your cleaned data instantly.

In [ ]:
# QUICK LOAD - Use this code in future sessions to reload your cleaned data

import pickle
import pandas as pd
import os
from pathlib import Path

# Detect if running in Google Colab
try:
    from google.colab import drive
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

# Set data folder path
if IN_COLAB:
    data_folder = '/content/drive/My Drive/USGS_Water_Data'
else:
    data_folder = str(Path('c:/Water_research/USGS_Downloads'))

# Load master dataframe from pickle (fastest method)
master_pickle_path = os.path.join(data_folder, 'master_dataframe_clean.pkl')
with open(master_pickle_path, 'rb') as f:
    master_df_clean = pickle.load(f)
print(f"✓ Loaded master DataFrame: {master_df_clean.shape} rows")

# Load individual site dataframes
sites_pickle_path = os.path.join(data_folder, 'site_dataframes_clean.pkl')
with open(sites_pickle_path, 'rb') as f:
    site_dataframes_clean = pickle.load(f)
print(f"✓ Loaded {len(site_dataframes_clean)} site DataFrames")

# OR: Load from CSV if you modified the data in Excel
# master_df_clean = pd.read_csv(os.path.join(data_folder, 'master_dataframe_clean.csv'))

print("\nYou're all set! Your cleaned water data is loaded and ready for analysis or modeling.")